# Data Collection Workbook

In [3]:
import yfinance as yf
import pandas as pd
from tqdm import tqdm

In [13]:
with open("./data/tickers.txt", "r") as f:
    tickers = [line.strip() for line in f if line.strip()]

### Download Candlestick data
raw = yf.download(
    tickers=tickers,
    period="25y",
    interval="1d",
    group_by="ticker",
    auto_adjust=False,
    threads=True,
    progress=True
)

dfs = []

for ticker in tickers:
    try:
        df = raw[ticker].copy()
        df["Ticker"] = ticker
        df.reset_index(inplace=True)
        dfs.append(df)
    except KeyError:
        print(f"Missing price data for {ticker}")

stock_df = pd.concat(dfs, ignore_index=True)

[**********************76%***********            ]  76 of 100 completed$MMC: possibly delisted; no price data found  (period=25y) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  100 of 100 completed

1 Failed download:
['MMC']: possibly delisted; no price data found  (period=25y) (Yahoo error = "No data found, symbol may be delisted")


In [14]:
# Get sector and industry information
sector_data = []

for ticker in tqdm(tickers, desc="Getting company info"):
    try:
        info = yf.Ticker(ticker).info

        sector_data.append({
            "Ticker": ticker,
            "Sector": info.get("sector"),
            "Industry": info.get("industry")
        })

    except Exception as e:
        print(f"Failed for {ticker}: {e}")

        sector_data.append({
            "Ticker": ticker,
            "Sector": None,
            "Industry": None
        })

sector_df = pd.DataFrame(sector_data)

Getting company info: 100%|██████████| 100/100 [00:19<00:00,  5.10it/s]


In [15]:
### Combine Data
stock_df = stock_df.merge(
    sector_df,
    on="Ticker",
    how="left"
)

stock_df = stock_df.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

stock_df.to_csv("./data/stock_data_with_sectors.csv", index=False)

In [16]:
stock_df

,Date,Open,High,Low,Close,Adj Close,Volume,Ticker,Sector,Industry
0,2001-06-04,0.376429,0.376964,0.365357,0.368929,0.309233,281920800.0,AAPL,Technology,Consumer Electronics
1,2001-06-05,0.371429,0.376786,0.363393,0.373929,0.313424,471794400.0,AAPL,Technology,Consumer Electronics
2,2001-06-06,0.373750,0.373750,0.363036,0.370179,0.310281,223176800.0,AAPL,Technology,Consumer Electronics
3,2001-06-07,0.369821,0.387500,0.365179,0.386786,0.324201,325180800.0,AAPL,Technology,Consumer Electronics
4,2001-06-08,0.386607,0.386607,0.369821,0.380714,0.319111,342624800.0,AAPL,Technology,Consumer Electronics
...,...,...,...,...,...,...,...,...,...,...
628495,2026-05-26,80.779999,81.309998,79.949997,80.230003,80.230003,5696600.0,ZTS,Healthcare,Drug Manufacturers - Specialty & Generic
628496,2026-05-27,80.019997,81.139999,78.730003,79.480003,79.480003,7874000.0,ZTS,Healthcare,Drug Manufacturers - Specialty & Generic
628497,2026-05-28,79.000000,80.199997,78.250000,78.269997,78.269997,7686200.0,ZTS,Healthcare,Drug Manufacturers - Specialty & Generic
628498,2026-05-29,78.510002,78.879997,77.440002,77.690002,77.690002,9295100.0,ZTS,Healthcare,Drug Manufacturers - Specialty & Generic
